# Reinforcement Learning–Based Active Learning for Semantic Segmentation

This notebook demonstrates a **reinforcement learning (REINFORCE)–based active learning pipeline**
for semantic segmentation using a U-Net backbone.

Unlike heuristic query strategies (entropy, diversity, etc.), the querying policy is **learned**
to directly optimize downstream segmentation performance.

### Goals
- Run a full RL-based active learning loop
- Analyze learning dynamics of the policy
- Compare against heuristic active learning baselines

### Key characteristics
- Policy-gradient (REINFORCE) training
- State = CNN bottleneck features + uncertainty measures
- Reward = ΔF1-score on validation set

## 1. Imports and Environment Setup

In [ ]:
import sys
from pathlib import Path

# Make src importable
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from config.config import ActiveLearningConfig
from src.utils import set_seed
from src.rl_active_learning import ActiveLearningSystemRL


## 2. Experiment Configuration

In [ ]:
config = ActiveLearningConfig.from_yaml(
    "../experiments/configs/rl_active_learning.yaml"
)

set_seed(config.seed)

print(config)

## 3. RL Active Learning System Initialization


In [ ]:
al_rl = ActiveLearningSystemRL(config)

## 4. Run experiment

In [ ]:
history = al_rl.run()
len(history), history[-1]

## 5. Visualize the history

In [ ]:
df = pd.DataFrame(history)
df


## 6. Validation performance over AL cycles


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df["dice"], "-o", label="Dice")
plt.plot(df["mean_iou"], "-o", label="IoU")
plt.xlabel("AL Cycle")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.show()

### RL Reward evolution

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(df["reward"], "-o")
plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("AL Cycle")
plt.ylabel("Reward (Δ Dice)")
plt.grid(True)
plt.show()

### Performance vs labeled samples


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(df["labeled"], df["dice"], "-o")
plt.xlabel("Labeled samples")
plt.ylabel("Dice")
plt.grid(True)
plt.show()

### Save results

In [ ]:
from src.utils import save_results_json

save_results_json(
    {
        "experiment_type": "rl_active_learning",
        "dataset": config.dataset_type,
        "seed": config.seed,
        "history": history,
        "final_metrics": history[-1],
    },
    experiment_name=f"rl_al_{config.dataset_type}_seed{config.seed}",
    results_dir="../results"
)


The system maintains:
- a frozen **oracle model** for feature extraction and uncertainty
- a **main model** trained on the growing labeled set
- a **policy network** trained with REINFORCE to select informative samples
